# 🧠 Procesamiento del Lenguaje Natural: De Bag of Words a Embeddings y Búsqueda Vectorial

**Curso:** Introducción a la Inteligencia Artificial Generativa  
**Módulo 2:** Procesamiento del Lenguaje y Bases de Datos Vectoriales  
**Instructor de referencia:** Daniel Betancur

---

### 🎯 Objetivos de aprendizaje de este notebook:
1. **Comprender el problema fundamental del PLN**: ¿Cómo convertir texto humano en números que una máquina pueda entender?
2. **Vectorización Clásica / Léxica**:
   - **Bag of Words (BoW)**: Representación por frecuencia de términos, ventajas y limitaciones críticas (pérdida de orden, ignorancia semántica).
   - **TF-IDF**: Ponderación estadística de especificidad e importancia léxica.
3. **Embeddings Semánticos (Open Source)**:
   - Representaciones densas en espacios vectoriales continuos con `sentence-transformers`.
   - Comprensión de cómo los modelos de lenguaje codifican el significado global de oraciones.
4. **Visualización del Espacio Vectorial en 2D**:
   - Reducción de dimensiones mediante **PCA (Principal Component Analysis)**.
   - Representación gráfica de clusters semánticos (animales, comida, tecnología, clima).
5. **Búsqueda Vectorial por Similitud (Semantic Search)**:
   - Cálculo matemático de **Similitud Coseno**, Distancia Euclidiana y Producto Punto.
   - Implementación de un motor de búsqueda semántica contra una base de conocimiento.


## 📦 1. Instalación y Carga de Dependencias

Para este módulo utilizaremos librerías clave del ecosistema de Data Science y NLP:
- `scikit-learn`: Para BoW (`CountVectorizer`), TF-IDF (`TfidfVectorizer`), reducción de dimensiones (`PCA`) y métricas (`cosine_similarity`).
- `sentence-transformers`: Librería open source estándar para generar embeddings de oraciones utilizando modelos preentrenados de Hugging Face.
- `matplotlib` y `seaborn`: Para la visualización gráfica en el plano 2D.
- `pandas` y `numpy`: Para manipulación tabular y álgebra lineal con vectores.


In [ ]:
# Descomenta y ejecuta la siguiente línea si necesitas instalar los paquetes:
# !pip install scikit-learn sentence-transformers matplotlib seaborn pandas numpy


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración estética para gráficos
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11

print("✅ Librerías base cargadas exitosamente.")


---
## 🧩 2. El Desafío del Significado: Representaciones Léxicas

> *"Los computadores procesan números, no entienden palabras con contexto y matices."*

Para que un algoritmo trabaje con texto, primero debemos **vectorizarlo**: transformar cadenas de caracteres en vectores numéricos dentro de un espacio vectorial.

Tradicionalmente se utilizaban **representaciones dispersas (sparse representations)** basadas en palabras clave (léxico).

Veamos cuatro textos de prueba clave:
1. `Doc 1`: *"El perro persigue al gato"*
2. `Doc 2`: *"El gato persigue al perro"*
3. `Doc 3`: *"Un cachorro corre detrás de un felino"*
4. `Doc 4`: *"El ingeniero programa software con inteligencia artificial"*

Observa dos desafíos fundamentales:
- `Doc 1` y `Doc 2` usan **exactamente las mismas palabras**, pero el significado es inverso (el sujeto y el objeto cambiaron de rol).
- `Doc 1` y `Doc 3` describen conceptualmente **la misma escena animal**, pero **no comparten vocabulario exacto** (*perro/cachorro*, *gato/felino*, *persigue/corre detrás*).


### 2.1 Bag of Words (BoW - Bolsa de Palabras)

El modelo de **Bolsa de Palabras (BoW)**:
1. Construye un vocabulario con todas las palabras únicas del corpus.
2. Cada documento se representa como un vector cuya dimensión es el tamaño del vocabulario.
3. Cada valor del vector representa cuántas veces aparece cada palabra en ese documento.

Implementémoslo con `CountVectorizer` de Scikit-Learn:


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

documentos_ejemplo = [
    "El perro persigue al gato",
    "El gato persigue al perro",
    "Un cachorro corre detrás de un felino",
    "El ingeniero programa software con inteligencia artificial"
]

# Vectorizador BoW
vectorizador_bow = CountVectorizer()
matriz_bow = vectorizador_bow.fit_transform(documentos_ejemplo)
vocabulario = vectorizador_bow.get_feature_names_out()

# Creamos un DataFrame para ver la matriz término-documento con claridad
df_bow = pd.DataFrame(
    matriz_bow.toarray(),
    index=[f"Doc {i+1}: '{doc[:26]}...'" for i, doc in enumerate(documentos_ejemplo)],
    columns=vocabulario
)

print(f"Tamaño del vocabulario: {len(vocabulario)} términos únicos")
df_bow


#### 🔍 Diagnóstico de Bag of Words:
1. **Pérdida absoluta del orden sintáctico**: Fíjate en `Doc 1` y `Doc 2`. Las filas son **exactamente idénticas**. Para BoW, *"el perro persigue al gato"* y *"el gato persigue al perro"* son el mismo vector.
2. **Cero comprensión semántica (sinonimia)**: `Doc 3` (*cachorro/felino*) no tiene ninguna coincidencia de palabras con `Doc 1` (*perro/gato*). Su similitud matemática es cero (`0.0`).
3. **Matriz dispersa (Sparse Matrix)**: Si tuviéramos un corpus de 10,000 artículos, el vector tendría más de 50,000 dimensiones y el 99.9% de los valores serían ceros.


### 2.2 TF-IDF (Term Frequency - Inverse Document Frequency)

¿Qué pasa con palabras como *"el"*, *"de"* o *"un"*? Aparecen en casi todos los documentos pero no aportan información sobre el tema central del texto.

**TF-IDF** mejora BoW aplicando dos factores:
1. **TF (Term Frequency)**: Frecuencia con la que aparece el término $t$ dentro del documento $d$.
2. **IDF (Inverse Document Frequency)**: Mide qué tan raro es el término en todo el corpus $D$:
   $$\text{IDF}(t, D) = \log\left(\frac{1 + N}{1 + |\{d \in D : t \in d\}|}\right) + 1$$
3. **Valor final ponderado**:
   $$\text{TF-IDF}(t, d, D) = \text{TF}(t, d) \times \text{IDF}(t, D)$$

- Si una palabra aparece en **muchos documentos**, su IDF disminuye hacia cero.
- Si una palabra aparece en **un solo documento específico** (ej. *"inteligencia"*), su peso TF-IDF es muy alto.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizador_tfidf = TfidfVectorizer()
matriz_tfidf = vectorizador_tfidf.fit_transform(documentos_ejemplo)

df_tfidf = pd.DataFrame(
    matriz_tfidf.toarray(),
    index=[f"Doc {i+1}" for i in range(len(documentos_ejemplo))],
    columns=vectorizador_tfidf.get_feature_names_out()
)

print("Matriz TF-IDF (pesos estadísticos ponderados):")
df_tfidf.round(3)


#### 💡 Conclusión sobre TF-IDF:
Aunque TF-IDF pondera muy bien la relevancia estadística de las palabras:
- **Sigue siendo puramente léxico**: `Doc 1` y `Doc 2` continúan teniendo exactamente el mismo vector TF-IDF.
- Sigue sin asociar *"cachorro"* con *"perro"*.
- Para capturar el significado, necesitamos pasar de la coincidencia de palabras a los **Embeddings**.


---
## 🚀 3. Embeddings: Codificando Significado en Vectores Densos

> *"Buscamos representaciones que capturen el significado y las relaciones entre palabras y frases."*

Los **Embeddings** representan un cambio de paradigma radical:
- **Vectores densos**: Cada dimensión es un número real continuo (no solo ceros y unos).
- **Espacio continuo de baja dimensionalidad**: Típicamente entre 384 y 1536 dimensiones fijas.
- **Comprensión contextual**: Dos textos con palabras completamente diferentes que expresan la misma idea tendrán vectores muy cercanos en el espacio vectorial.

Usaremos un modelo **Open Source** de vanguardia desde Hugging Face mediante la librería `sentence-transformers`:
- **Modelo:** `paraphrase-multilingual-MiniLM-L12-v2`
- **Características:** Ligero (~120 MB), muy rápido en CPU, entrenado en más de 50 idiomas (incluyendo español) y con **384 dimensiones**.


In [ ]:
from sentence_transformers import SentenceTransformer

print("📥 Descargando/Cargando modelo de embeddings open-source...")
modelo_embeddings = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
print(f"✅ Modelo cargado exitosamente: {modelo_embeddings}")


In [ ]:
# Inspeccionemos el embedding de una oración de prueba
oracion = "El perro persigue al gato"
vector = modelo_embeddings.encode(oracion)

print(f"Texto: '{oracion}'")
print(f"Dimensión del vector (número de características): {vector.shape[0]}")
print(f"Tipo de datos: {vector.dtype}")
print(f"Primeros 10 valores continuos del vector:\n{vector[:10].round(4)}")
print(f"Norma Euclidiana del vector: {np.linalg.norm(vector):.4f}")


### 3.1 Prueba de Fuego: Embeddings frente a los Casos de BoW

Generemos los embeddings para nuestras 4 frases de prueba y calculemos su matriz de **Similitud Coseno**:


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

frases_test = [
    "El perro persigue al gato",
    "El gato persigue al perro",
    "Un cachorro corre detrás de un felino",
    "La bolsa de valores cerró con ganancias en Wall Street"
]

# Generamos los embeddings densos
vectores_test = modelo_embeddings.encode(frases_test)

# Calculamos la matriz de similitud coseno entre todos los pares
matriz_sim = cosine_similarity(vectores_test)

df_sim = pd.DataFrame(
    matriz_sim,
    index=[f"[{i+1}] {f[:22]}..." for i, f in enumerate(frases_test)],
    columns=[f"[{i+1}] {f[:22]}..." for i, f in enumerate(frases_test)]
)

print("Matriz de Similitud Coseno con Embeddings:")
df_sim.round(4)


#### 🌟 Hallazgos clave:
1. **Reconocimiento del cambio de sujeto/objeto**: A diferencia de BoW que daba $1.0$, los embeddings diferencian la frase 1 de la frase 2 (similitud alta ~0.89 pero **distinta de 1.0**).
2. **Captura semántica y sinonimia**: La frase 1 (*perro persigue al gato*) y la frase 3 (*cachorro corre detrás de un felino*) tienen una **similitud muy alta** (~0.75 - 0.85), demostrando que el modelo comprende la equivalencia de conceptos sin requerir palabras idénticas.
3. **Discriminación conceptual**: La frase 4 (*bolsa de valores*) tiene una similitud muy baja (<0.15) con las frases de animales.


---
## 🗺️ 4. Visualización de Embeddings en el Espacio Vectorial 2D

Nuestros embeddings habitan en un espacio de **384 dimensiones**.
Para poder verlos como humanos en una pantalla, aplicamos **PCA (Análisis de Componentes Principales)** para proyectar las 384 dimensiones en **2 dimensiones principales**, preservando la mayor cantidad de información y distancia relativa posible.

Vamos a crear un corpus con frases pertenecientes a 4 temas distintos para observar cómo se agrupan geométricamente:
1. 🐶 **Mascotas y Animales**
2. 🍕 **Gastronomía y Comida**
3. 💻 **Tecnología e Inteligencia Artificial**
4. 🏖️ **Clima y Vacaciones**


In [ ]:
corpus_categorias = {
    "Animales": [
        "El perro persigue al gato en el parque",
        "Un cachorro juega alegre con una pelota",
        "El felino duerme plácidamente sobre el sofá",
        "Los gatos cazan pequeños ratones por instinto"
    ],
    "Gastronomía": [
        "Me fascina comer una pizza napolitana con queso",
        "El restaurante italiano prepara pasta artesanal deliciosa",
        "Receta casera para cocinar hamburguesas con papas",
        "Un buen café espresso caliente por la mañana"
    ],
    "Tecnología / IA": [
        "Los modelos de lenguaje aprenden representaciones semánticas",
        "La inteligencia artificial generativa transformará la industria",
        "Desarrollo de software con redes neuronales y deep learning",
        "Entrenando algoritmos de machine learning con grandes volúmenes de datos"
    ],
    "Clima / Vacaciones": [
        "Hoy hace un día muy soleado y caluroso en la playa",
        "El pronóstico del tiempo anuncia lluvias fuertes esta tarde",
        "Viajaremos a la costa del Caribe durante las vacaciones",
        "La temperatura en la alta montaña desciende bajo cero"
    ]
}

textos_corpus = []
categorias_corpus = []
colores = {
    "Animales": "#E63946",
    "Gastronomía": "#F4A261",
    "Tecnología / IA": "#2A9D8F",
    "Clima / Vacaciones": "#457B9D"
}

for cat, frases in corpus_categorias.items():
    for f in frases:
        textos_corpus.append(f)
        categorias_corpus.append(cat)

print(f"Total de frases para visualizar: {len(textos_corpus)}")


In [ ]:
from sklearn.decomposition import PCA

# 1. Generar los embeddings de 384 dimensiones
embeddings_todos = modelo_embeddings.encode(textos_corpus)
print(f"Matriz de embeddings calculada: {embeddings_todos.shape}")

# 2. Reducción a 2 dimensiones mediante PCA
pca = PCA(n_components=2, random_state=42)
embeddings_2d = pca.fit_transform(embeddings_todos)

varianza = pca.explained_variance_ratio_
print(f"Varianza explicada: Dim 1 = {varianza[0]*100:.1f}%, Dim 2 = {varianza[1]*100:.1f}% (Total: {sum(varianza)*100:.1f}%)")


In [ ]:
# 3. Gráfico del espacio vectorial 2D
fig, ax = plt.subplots(figsize=(13, 8))

df_vis = pd.DataFrame({
    "x": embeddings_2d[:, 0],
    "y": embeddings_2d[:, 1],
    "categoria": categorias_corpus,
    "texto": textos_corpus
})

# Graficar cada categoría
for cat, color in colores.items():
    sub = df_vis[df_vis["categoria"] == cat]
    ax.scatter(
        sub["x"], sub["y"], 
        c=color, label=cat, s=140, alpha=0.9, edgecolors="black", linewidth=1.2
    )

# Dibujar vectores desde el origen (0, 0) para ilustrar el concepto de vector espacial
for _, row in df_vis.iterrows():
    ax.annotate(
        "", xy=(row["x"], row["y"]), xytext=(0, 0),
        arrowprops=dict(arrowstyle="->", color="gray", alpha=0.25, lw=1)
    )

# Etiquetas con texto acortado
for _, row in df_vis.iterrows():
    etiqueta = row["texto"] if len(row["texto"]) <= 28 else row["texto"][:26] + "..."
    ax.annotate(
        etiqueta,
        xy=(row["x"], row["y"]),
        xytext=(6, 5),
        textcoords="offset points",
        fontsize=9,
        fontweight="semibold",
        bbox=dict(boxstyle="round,pad=0.25", fc="white", ec=colores[row["categoria"]], alpha=0.85)
    )

# Configuración de ejes y títulos
ax.axhline(0, color="black", linestyle="--", linewidth=0.8, alpha=0.4)
ax.axvline(0, color="black", linestyle="--", linewidth=0.8, alpha=0.4)
ax.set_title("Visualización del Espacio Vectorial Semántico (Proyección 2D con PCA)", fontsize=14, fontweight="bold", pad=15)
ax.set_xlabel("Dimensión Principal 1 (PCA)", fontsize=11)
ax.set_ylabel("Dimensión Principal 2 (PCA)", fontsize=11)
ax.legend(title="Categoría Semántica", loc="upper right", frameon=True, fontsize=10)

plt.tight_layout()
plt.show()


#### 💡 ¿Qué observamos en el espacio 2D?
1. **Clusters naturales**: Las frases sobre animales se agrupan en su propio vecindario; las de comida en otro, y así sucesivamente.
2. **Orientación geométrica**: Las frases afines apuntan en ángulos similares respecto al origen (0,0), lo que confirma por qué la **similitud coseno** es la métrica reina para comparar significado.


---
## 🔍 5. Búsqueda Vectorial por Similitud (Semantic Search)

> *"Una vez tenemos una colección de vectores, ¿cómo encontramos aquellos que son más relevantes para resolver una consulta?"*

### 5.1 Las 3 Métricas de Similitud y Distancia

| Métrica | Fórmula | Interpretación |
| :--- | :--- | :--- |
| **Similitud Coseno** | $$\cos(\theta) = \frac{\mathbf{u} \cdot \mathbf{v}}{\|\mathbf{u}\| \|\mathbf{v}\|}$$ | Mide el **ángulo** entre dos vectores. Rango $[-1, 1]$. Es la métrica estándar en PLN porque ignora la longitud del texto y mide puramente la orientación. |
| **Distancia Euclidiana ($L_2$)** | $$d(\mathbf{u}, \mathbf{v}) = \sqrt{\sum_{i=1}^n (u_i - v_i)^2}$$ | Mide la distancia en línea recta entre las puntas de los vectores. |
| **Producto Punto (Dot Product)** | $$\mathbf{u} \cdot \mathbf{v} = \sum_{i=1}^n u_i v_i$$ | Combina orientación y magnitud. Si los vectores están normalizados ($\|\mathbf{u}\|=1$), es idéntico al Coseno. |

Calculemos manualmente estas 3 métricas en código:


In [ ]:
def calcular_distancias_manual(vec_a, vec_b):
    """Calcula Similitud Coseno, Distancia Euclidiana y Producto Punto."""
    # 1. Producto Punto
    dot = np.dot(vec_a, vec_b)
    
    # 2. Normas L2
    norm_a = np.linalg.norm(vec_a)
    norm_b = np.linalg.norm(vec_b)
    
    # 3. Similitud Coseno
    cos_sim = dot / (norm_a * norm_b)
    
    # 4. Distancia Euclidiana
    l2_dist = np.linalg.norm(vec_a - vec_b)
    
    return {
        "Similitud Coseno (mayor es más cerca)": cos_sim,
        "Distancia Euclidiana (menor es más cerca)": l2_dist,
        "Producto Punto": dot
    }

# Ejemplo con dos frases afines vs una no afín
v_gato = modelo_embeddings.encode("Un gatito pequeño jugando en el jardín")
v_felino = modelo_embeddings.encode("El felino corre velozmente sobre la hierba")
v_finanzas = modelo_embeddings.encode("La tasa de inflación mensual subió un punto porcentual")

comp_cercanas = calcular_distancias_manual(v_gato, v_felino)
comp_lejanas = calcular_distancias_manual(v_gato, v_finanzas)

df_metricas = pd.DataFrame([comp_cercanas, comp_lejanas], index=[
    "Gatito vs Felino (Temática afín)",
    "Gatito vs Inflación (Temática dispar)"
])
df_metricas.round(4)


### 5.2 Implementación de un Buscador Semántico Paso a Paso

Vamos a crear una base de conocimiento documental y desarrollar una función de búsqueda que:
1. Reciba cualquier consulta en lenguaje natural libre (query).
2. Convierta la query en un embedding con el modelo.
3. Calcule la similitud coseno contra todos los documentos indexados.
4. Devuelva y grafique el **Top-K** de documentos más cercanos semánticamente.


In [ ]:
# Base de conocimiento (Catálogo documental)
base_conocimiento = [
    "Python es el lenguaje de programación predilecto para inteligencia artificial y ciencia de datos.",
    "Los perros de raza golden retriever son animales de compañía dóciles y muy inteligentes.",
    "La auténtica pizza napolitana se hornea a leña con masa madre, tomate y queso mozzarella.",
    "Las redes neuronales profundas y transformers revolucionaron el procesamiento del lenguaje.",
    "Para preparar un café espresso equilibrado se requiere molienda fina y adecuada presión de agua.",
    "El telescopio espacial capta radiación infrarroja de galaxias formadas tras el Big Bang.",
    "El felino doméstico pasa varias horas del día descansando y cazando presas pequeñas.",
    "El cambio climático global genera aumentos de temperatura en los arrecifes de coral marinos."
]

# Indexamos los documentos en la base vectorial (generar y guardar embeddings)
embeddings_base = modelo_embeddings.encode(base_conocimiento)
print(f"✅ Base de conocimiento indexada: {len(base_conocimiento)} documentos en {embeddings_base.shape[1]} dimensiones.")


In [ ]:
def buscar_semantico(query: str, top_k: int = 3, graficar: bool = True):
    """
    Ejecuta una búsqueda vectorial por similitud coseno contra la base de conocimiento.
    """
    # 1. Vectorizar la consulta del usuario
    vec_query = modelo_embeddings.encode([query])
    
    # 2. Calcular la similitud coseno contra toda la base de conocimiento
    similitudes = cosine_similarity(vec_query, embeddings_base)[0]
    
    # 3. Obtener los índices de los K documentos más cercanos (orden descendente)
    mejores_indices = np.argsort(similitudes)[::-1][:top_k]
    
    print(f"\n🔎 CONSULTA: \"{query}\"")
    print("=" * 75)
    
    resultados = []
    for ranking, idx in enumerate(mejores_indices, 1):
        score = similitudes[idx]
        doc_texto = base_conocimiento[idx]
        print(f"[{ranking}] Similitud: {score:.4f} | Doc #{idx + 1}:")
        print(f"    \"{doc_texto}\"")
        resultados.append({"rank": ranking, "id": idx + 1, "score": score, "texto": doc_texto})
    
    # Graficar resultados
    if graficar:
        plt.figure(figsize=(9, 3.5))
        etiquetas = [f"Doc #{r['id']}" for r in resultados]
        puntajes = [r['score'] for r in resultados]
        paleta = sns.color_palette("Blues_r", n_colors=top_k)
        
        barras = plt.barh(etiquetas[::-1], puntajes[::-1], color=paleta[::-1], edgecolor="black", alpha=0.85)
        plt.xlim(0, 1.0)
        plt.xlabel("Similitud Coseno", fontweight="bold")
        plt.title(f"Top-{top_k} Resultados para: '{query}'", fontsize=12, fontweight="bold")
        
        for b in barras:
            ancho = b.get_width()
            plt.text(ancho + 0.02, b.get_y() + b.get_height()/2, f"{ancho:.3f}", 
                     va="center", fontweight="bold", fontsize=10)
        
        plt.tight_layout()
        plt.show()
        
    return resultados


### 5.3 Pruebas de Búsqueda Semántica

Probemos consultas con palabras que **no existen de forma literal** en los documentos, pero que guardan relación semántica evidente:


In [ ]:
# Consulta 1: Animales de compañía
_ = buscar_semantico("Mascotas cariñosas para la familia", top_k=3)


In [ ]:
# Consulta 2: Gastronomía italiana
_ = buscar_semantico("Platillos típicos con masa y queso", top_k=3)


In [ ]:
# Consulta 3: Programación y Deep Learning
_ = buscar_semantico("Desarrollo de algoritmos de aprendizaje de máquinas", top_k=3)


### 5.4 Comparativa: Búsqueda Tradicional de Palabras Clave (BoW) vs Búsqueda Semántica

¿Qué respondería un motor de búsqueda léxico tradicional basado en coincidencia exacta de palabras frente a nuestra consulta?


In [ ]:
def buscar_bow_comparativa(query: str, top_k: int = 3):
    """Simula búsqueda tradicional léxica por coincidencia de palabras exactas."""
    cv = CountVectorizer().fit(base_conocimiento)
    kb_bow = cv.transform(base_conocimiento)
    q_bow = cv.transform([query])
    
    sims = cosine_similarity(q_bow, kb_bow)[0]
    top_idx = np.argsort(sims)[::-1][:top_k]
    
    print(f"\n🔤 RESULTADOS BÚSQUEDA LÉXICA (Bag of Words / Palabras clave): '{query}'")
    for rank, idx in enumerate(top_idx, 1):
        print(f"  [{rank}] Score BoW: {sims[idx]:.4f} -> Doc #{idx+1}: \"{base_conocimiento[idx][:65]}...\"")

query_comparativa = "Mascotas cariñosas para la familia"
print("=" * 75)
buscar_bow_comparativa(query_comparativa)
print("=" * 75)
print("🧠 RESULTADOS BÚSQUEDA SEMÁNTICA (Embeddings):")
_ = buscar_semantico(query_comparativa, top_k=2, graficar=False)


#### 💥 La revelación práctica:
- En la búsqueda léxica (BoW), el score es **0.0000** para todos los documentos porque las palabras *"mascotas"* o *"cariñosas"* no aparecen literalmente escritas.
- En cambio, la **búsqueda semántica con embeddings** identificó con un score sobresaliente (~0.68) el documento de los perros golden retriever (*"animales de compañía dóciles y muy inteligentes"*).


---
## 🏁 6. Conclusiones y Conexión con Bases de Datos Vectoriales (VDB)

Hemos recorrido la evolución del procesamiento del lenguaje natural:

| Enfoque | Representación | Significado Semántico | Manejo de Paráfrasis | Espacio Vectorial |
| :--- | :--- | :--- | :--- | :--- |
| **Bag of Words** | Dispersa (*Sparse*) | ❌ Nulo (solo cuenta palabras) | ❌ Inexistente | Dimensión = Vocabulario ($10^4 - 10^6$) |
| **TF-IDF** | Dispersa ponderada | ❌ Estadístico / Léxico | ❌ Inexistente | Dimensión = Vocabulario ($10^4 - 10^6$) |
| **Embeddings** | Densa (*Dense*) | ✅ Comprensión contextual profunda | ✅ Excelente | Dimensión fija ($384 - 1536$) |

### ⚡ ¿Por qué necesitamos Bases de Datos Vectoriales (VDB)?
En este notebook calculamos la similitud coseno de forma **exhaustiva ($O(N)$)** contra 8 documentos en microsegundos.

Sin embargo, en sistemas reales (como aplicaciones RAG o motores de búsqueda empresariales):
- Las bases de datos contienen **millones de documentos**.
- Comparar una consulta contra millones de vectores en tiempo real mediante fuerza bruta es inviable computacionalmente.

Para resolver esto, las **Bases de Datos Vectoriales (como Chroma, Pinecone, Milvus, Weaviate o Qdrant)** implementan algoritmos de **Búsqueda de Vecinos Más Cercanos Aproximados (ANN - Approximate Nearest Neighbors)** como HNSW (*Hierarchical Navigable Small World*), permitiendo encontrar los documentos más cercanos en cuestión de **milisegundos ($O(\log N)$)**.

---
### ✍️ Ejercicio Práctico para el Estudiante:
1. Agrega 3 oraciones nuevas a `base_conocimiento` sobre un dominio de tu preferencia (ej. medicina, deportes o astronomía).
2. Diseña 2 consultas que usen sinónimos y lenguaje coloquial.
3. Ejecuta `buscar_semantico` y evalúa si el ranking y los scores reflejan la cercanía del concepto.

---
*Notebook desarrollado para el curso Introducción a la Inteligencia Artificial Generativa.*  
*Universidad Nacional de Colombia - Módulo 2.*
